<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11.1-camera-navigation/Ex11.1_cnn_steering_from_a_frame.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11.1 — Steering from a Frame

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L11.1 · Vision-Based Navigation**

A car follows a line of tape on a floor. Its camera gives a frame, and the
policy has to say where the line is a little way ahead: one number, from which
the steering follows. You find that number two ways - a classical detector
that looks for bright pixels, and a convolutional network trained on labelled
frames - and compare them on accuracy and on time. Then you see what the
lecture calls the real constraint: how far the car drives between two
decisions, and what storing the network in fewer bits costs.

This notebook runs on Colab or a laptop. **Its frames are drawn by
`problem.py`, not taken by a camera**, which is what makes the true answer
known. The set's other notebook, `Ex11.1_on_the_car.ipynb`, collects real
frames on your own car and drives the scored laps.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and the geometry of the loop | — |
| **4** | the reference, and the classical detector at a resolution for an agreed accuracy | — |
| **5** | a convolutional network, trained on two datasets, and the comparison | the network, the training step |
| **6** | the loop rate and the blind distance; the weights in fewer bits | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About fifteen seconds on a CPU. Numbers are printed with at most two decimals,
except errors smaller than that.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11.1-camera-navigation/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex11.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The frame is 48 × 64 grey pixels: the floor near the car at the bottom, the
floor further away at the top. Wanted: the position of the tape in the row
that is 0.6 of the way up the frame, as a number between −1 (the left edge)
and +1 (the right edge). That number is the **target**. It is a regression,
not a classification (L11.1's *Road Following as Regression*).

## 2 · The data

| | |
|---|---|
| frame | 48 × 64 pixels, grey, values 0 to 1 |
| floor | brightness 0.35, with a slope across the frame of up to ±0.15: light from one side |
| tape | 0.55 brighter than the floor, about 2 pixels wide where the target is read |
| noise | 0.04 on every pixel |
| glare | on some frames, a bright patch beside the line: a window reflected in the floor |
| what moves the line | the car's offset from it, the car's heading, and the curve of the road |

**The frames are synthetic.** On the car the label is a click on a real
frame; here the frame is drawn from three numbers, so the true target is
known exactly. Three ways of collecting frames are used below:

| collection | the car is | glare |
|---|---|---|
| `line` | on the line and pointing along it | none |
| `off-line` | up to 0.6 of the half-frame beside it, and turned | none |
| `glare` | as off-line | on every frame |

## 3 · The geometry of the loop

**From the target to the steering.** The target is a point ahead of the car,
at an angle $\alpha$ to its heading and a distance $\ell$ away. A car of
wheelbase $L$ reaches it on a circle if it steers by

$$\delta = \arctan\!\Big(\frac{2L\sin\alpha}{\ell}\Big)$$

**Between two decisions the car is blind** (L11.1's *Latency, Not Accuracy*).
At speed $v$ and $f$ decisions a second it drives

$$d = \frac{v}{f}$$

without a new decision. The cell below prints the data and draws six frames
with their true target.

In [ ]:
pb.describe_problem()
fig, axes = plt.subplots(1, 6, figsize=(13, 2.2))
for i, kind in enumerate(("line", "line", "off-line", "off-line", "glare", "glare")):
    img, tgt = pb.frames(1, kind, seed=20 + i)                # one frame, and its true target
    axes[i].imshow(img[0, 0], cmap="gray", vmin=0, vmax=1)
    axes[i].plot((pb.W - 1) / 2 * (1 + tgt[0]), pb.ROW, "r+", ms=12, mew=2)   # the target, in pixels
    axes[i].set_title(f"{kind}: {tgt[0]:+.2f}", fontsize=9); axes[i].axis("off")
plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, and six frames with a red
cross on the tape in row 19. One pixel is 0.032 of the target's range. In the
last two frames there is a second bright thing, the glare, which is not the
line.

---

## 4 · The reference, and the classical detector

**The reference** is the true target: the frame was drawn from it.

**The classical way** needs no training. Take a few rows round the look-ahead
row, estimate the floor's brightness by the median, keep what is brighter than
the floor by 0.15, and return the brightness-weighted mean column.
`pb.detect(images)` does that.

**Its cost is set by the resolution.** `pb.shrink(images, k)` replaces every
$k \times k$ block of pixels by its mean. A smaller frame is cheaper to read
out and to process, and blurs the tape.

**The agreed accuracy is 0.03**: a mean error of one pixel of the full frame.
The cell below tests the detector at four resolutions on 1000 frames of each
collection.

In [ ]:
TEST = {kind: pb.frames(1000, kind, seed=100 + i) for i, kind in enumerate(("line", "off-line", "glare"))}   # held out: never trained on
AGREED = 0.03

def mean_error(predict, kind):
    x, y = TEST[kind]
    return float(np.mean(np.abs(predict(x) - y)))             # the mean distance from the true target

classic = {}
for k in (1, 2, 4, 8):
    classic[k] = [mean_error(lambda x: pb.detect(pb.shrink(x, k)), kind) for kind in TEST]
    ok = max(classic[k][:2]) <= AGREED
    print(f"  {pb.H // k:2d} x {pb.W // k:2d} pixels: line {classic[k][0]:.3f}, off-line {classic[k][1]:.3f}, glare {classic[k][2]:.3f}"
          + ("   <- meets 0.03 without glare" if ok else ""))
K = 4                                                         # the coarsest frame that meets it
frame1 = TEST["line"][0][:1]                                  # one frame, for the timing
t_classic = middle(lambda: pb.detect(pb.shrink(frame1, K)))
print(f"  the detector on one 12 x 16 frame: {1000 * t_classic:.2f} ms")

**What you should see.** Without glare the detector is within 0.002 at full
resolution and within 0.03 down to **12 × 16 pixels**; at 6 × 8 the tape is
blurred into the floor and it fails. It does not care where the car is: line
and off-line are the same to it. **With glare it is wrong by 0.25 to 0.46 at
every resolution**, a quarter of the half-frame: the glare is bright, and
bright is all it looks for.

---

## 5 · A convolutional network, and the comparison

**The network** (L11.1's *Why a Convolutional Network, and Not Any Other*)
takes the frame and returns the target. Three convolutional layers, each
halving the frame and doubling the channels, then two dense layers:

| layer | output |
|---|---|
| convolution 5 × 5, stride 2, 8 channels, ReLU | 8 × 24 × 32 |
| convolution 3 × 3, stride 2, 16 channels, ReLU | 16 × 12 × 16 |
| convolution 3 × 3, stride 2, 32 channels, ReLU | 32 × 6 × 8 |
| flatten, dense 32, ReLU | 32 |
| dense 1 | the target |

The car's own network is a pretrained ResNet-18 with a new head. This one is
small enough to train from nothing in a few seconds, and it is the same kind
of network.

**The loss** is the mean squared distance between the network's target and the
true one. There is no physics in it: this is supervised learning (L11.1's
*The One Lecture Without a PDE*).

**Two datasets, 3000 frames each.** One collected along the line only. One
collected **deliberately**: from beside the line and turned away from it, and
with glare on a third of the frames (L11.1's *Collecting Data Is the
Exercise*).

**TODO 1** is the network; **TODO 2** the training step.

In [ ]:
class Steer(torch.nn.Module):                                 # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        nn = torch.nn
        # TODO 1: the layers of the table above, in an nn.Sequential called self.body.
        #
        #   self.body = nn.Sequential(                            # the layers, applied one after the other
        #       nn.Conv2d(1, 8, 5, stride=2, padding=2), nn.ReLU(),     # convolution: 1 channel in, 8 out, the frame halved
        #       nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),    # convolution: 16 channels, halved again
        #       nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # convolution: 32 channels, 6 x 8 left
        #       nn.Flatten(),                                     # the 32 x 6 x 8 block as one row of 1536 numbers
        #       nn.Linear(32 * 6 * 8, 32), nn.ReLU(),             # dense layer: 1536 inputs -> 32 outputs
        #       nn.Linear(32, 1))                                 # dense layer: 32 inputs -> the target
        raise NotImplementedError("Write the network's layers")
    def forward(self, x):
        return self.body(x).squeeze(1)                        # one number per frame

In [ ]:
def train(kind, n=3000, epochs=12, batch=64):
    # a network trained on n frames collected the way `kind` says
    set_seed(88)                                              # fix the random start so a run repeats
    x, y = pb.frames(n, kind, seed=1)                         # the frames and their true targets
    X, Y = to_tensor(x).to(DEVICE), to_tensor(y).squeeze(1).to(DEVICE)   # NumPy arrays -> tensors on the device
    net = Steer().to(DEVICE)                                  # the network, on the device
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)         # Adam: adaptive steps for every weight
    t0 = time.perf_counter()
    for _ in range(epochs):                                   # one epoch: every frame seen once
        order = torch.randperm(n, device=DEVICE)              # the frames in a new random order
        for i in range(0, n, batch):
            b = order[i:i + batch]                            # one mini-batch of 64 frames
            # TODO 2: one training step on the mini-batch X[b], Y[b].
            #
            #   opt.zero_grad()                                   # clear the gradients of the step before
            #   loss = (net(X[b]) - Y[b]).pow(2).mean()           # mean squared distance from the true target
            #   loss.backward()                                   # backpropagation: the gradient of the loss
            #   opt.step()                                        # one Adam step on the weights
            raise NotImplementedError("Write the training step")
    return net, time.perf_counter() - t0

In [ ]:
net_line, sec_line = train("line")                            # collected along the line only
net, sec = train("deliberate")                                # collected deliberately
print(f"{parameter_count(net)} weights; trained in {sec_line:.0f} s and {sec:.0f} s")

def run(model):
    # the network as a function of NumPy frames, for mean_error
    def predict(x):
        with torch.no_grad():  # no gradient tracking: evaluation only
            return to_numpy(model(to_tensor(x).to(DEVICE)))   # frames -> tensor -> the network -> NumPy targets
    return predict

In [ ]:
err_line = [mean_error(run(net_line), kind) for kind in TEST]
err_net = [mean_error(run(net), kind) for kind in TEST]
x1 = to_tensor(frame1).to(DEVICE)                             # one frame, for the timing
with torch.no_grad():  # no gradient tracking: evaluation only
    t_net = middle(lambda: net(x1))

print(f"{'mean error on 1000 held-out frames':40s}{'line':>8s}{'off-line':>10s}{'glare':>8s}{'one frame':>12s}{'training':>10s}")
print(f"  {'classical detector, 12 x 16':38s}{classic[K][0]:8.3f}{classic[K][1]:10.3f}{classic[K][2]:8.3f}{1000 * t_classic:9.2f} ms{'-':>10s}")
print(f"  {'network, trained along the line':38s}{err_line[0]:8.3f}{err_line[1]:10.3f}{err_line[2]:8.3f}{1000 * t_net:9.2f} ms{sec_line:8.0f} s")
print(f"  {'network, trained deliberately':38s}{err_net[0]:8.3f}{err_net[1]:10.3f}{err_net[2]:8.3f}{1000 * t_net:9.2f} ms{sec:8.0f} s")

x, y = TEST["glare"]
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for a, (name, pred) in zip(ax, (("classical detector", pb.detect(pb.shrink(x, K))),
                                ("network, along the line", run(net_line)(x)), ("network, deliberately", run(net)(x)))):
    a.plot([-1, 1], [-1, 1], "k--", lw=1); a.plot(y, pred, ".", ms=2)
    a.set_title(name + ", frames with glare", fontsize=9); a.set_xlabel("true target")
ax[0].set_ylabel("predicted target"); plt.tight_layout()
plt.savefig(cc.output_path("Ex11.1_report.png"), dpi=150); plt.show()

**What you should see.** Three rows that fail in three different places.

* **The classical detector** is within 0.03 without glare and wrong by about
  0.37 with it.
* **The network trained along the line** is within 0.002 on frames like the
  ones it saw, and wrong by about 0.14 the moment the car is beside the line -
  which is exactly when the answer matters. It has never seen the line
  anywhere but in the middle.
* **The network trained deliberately** is within 0.01 on all three. Same
  network, same number of frames, same training time. Only the collection
  differs.

In the plots, the detector's points scatter far from the diagonal wherever
the glare outweighed the tape; the first network's flatten towards the
middle; the second's lie on it.

---

## 6 · The loop rate, and the weights in fewer bits

**The blind distance.** `pb.drive(f, v, error)` drives a bicycle model round
a stadium track of 7.20 m at speed $v$, with a steering decision $f$ times a
second from a target seen with the error measured above. Computing a decision
takes one period, so the wheels follow a command that is one frame old and
hold it until the next. The cell reports the largest distance from the
centreline. **The car's wheelbase (0.20 m), its steering limit (30°) and the
track are assumed, not measured.** Half the narrowest gap of the on-car
notebook's placeholder course is 17.5 cm.

**Fewer bits.** On the car the network is converted and its weights stored
in fewer bits, to make a decision take less time (L11.1's *TensorRT: Where the
Frame Rate Comes From*). `pb.round_weights(net, b)` rounds every weight onto
$2^b$ levels. The second half of the cell measures what that does to the
error. The network is a regression, so every rounding error lands in the
answer.

In [ ]:
noise = err_net[1]                                            # the deliberate network's error, off the line
print(f"{'speed':>8s}{'decisions':>12s}{'blind':>9s}{'furthest from the centreline':>32s}")
loop = []
for v, f in ((1.0, 30), (2.0, 30), (2.0, 20), (1.0, 10), (2.0, 15), (2.0, 12), (2.0, 10), (1.0, 5)):
    dev = pb.drive(f, v, noise)                               # two laps of the bicycle model
    loop.append((v / f, dev))
    print(f"{v:6.1f} m/s{f:9d} Hz{100 * v / f:7.1f} cm{100 * dev:27.2f} cm" + ("   <- off the course" if dev > pb.HALF_CLEARANCE else ""))

x, y = TEST["off-line"]
bits = (8, 6, 5, 4, 3, 2)
err_bits = [mean_error(run(pb.round_weights(net, b)), "off-line") for b in bits]   # the same network, its weights rounded
print("weights stored in " + ", ".join(f"{b}" for b in bits) + " bits: mean error " + ", ".join(f"{e:.3f}" for e in err_bits)
      + f"  (as trained: {err_net[1]:.3f})")
print(f"one frame through the network on this machine: {1000 * t_net:.2f} ms - not the car's Nano, and not the whole loop")

**What you should see.** The car stays within about 2 cm of the centreline
for every blind distance up to 10 cm, is 3 cm off at 13 cm, and leaves the
course at 17 cm and beyond. **Speed and loop rate do not matter separately**: 1 m/s at
10 Hz and 2 m/s at 20 Hz are both 10 cm blind, and both hold the line; 1 m/s
at 5 Hz and 2 m/s at 10 Hz are both 20 cm blind, and both leave it. The
weights can be stored in 5 bits with an error still about 0.01; at 3 bits the
error is about 0.15, and the network is no better than one trained along the
line.

---

## 7 · What the notebook says

* **The data decided the result, not the network.** Two identical networks,
  the same number of frames and the same training time: one fails beside the
  line and in glare, the other does not. Frames from where the car should not
  be are the ones that teach it to come back.
* **The classical detector needs no data and no training**, is the fastest of
  the three, and is exact on the frames it was designed for. It fails on what
  its designer did not think of. A better classical detector could reject the
  glare; someone has to think of that, too.
* **The error on held-out frames is only as good as the held-out frames.** The
  first network's error along the line was the smallest number in the table.
* **The blind distance is what the loop rate buys.** The same distance at two
  speeds gave the same result. A network twice as accurate and half as fast
  doubles it.
* **Fewer bits are a trade, to be measured.** This regression kept its
  accuracy down to 5 bits and lost it at 3. Whether the smaller network is
  also faster depends on the device, and only a measurement on the device says.
* **None of this is your car.** The frames are drawn, the track and the
  wheelbase are assumed, and the timings are a laptop's. The on-car notebook
  measures the loop rate that counts.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex11.1_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L11.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The policy returns one number per frame. Why a regression and not a classification, and why a convolutional network for it? (-> L11.1 Q1)": """
""",
    "The two networks differ only in the frames they were trained on. Say what the first one never saw, why its own held-out error did not show the problem, and what you will collect on the car because of it. (-> L11.1 Q2)": """
""",
    "Section 6 gave the same result for 1 m/s at 10 Hz and 2 m/s at 20 Hz. What is the quantity that decides, at what value did the car leave the course, and what are the two ways to reduce it? (-> L11.1 Q3)": """
""",
    "At how many bits did the network stop being usable, and what would you have to measure on the car before claiming that the converted network is faster? (-> L11.1 Q4)": """
""",
    "To conclude: the classical detector and the network each failed somewhere. For the course your car will drive, which would you start from, and what would you check before trusting it at twice the speed? (-> L11.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| mean error on 1000 held-out frames | line | off-line | glare | one frame | training |", "|---|---|---|---|---|---|",
            f"| classical detector, 12 x 16 | {classic[K][0]:.3f} | {classic[K][1]:.3f} | {classic[K][2]:.3f} | {1000 * t_classic:.2f} ms | - |",
            f"| network, trained along the line | {err_line[0]:.3f} | {err_line[1]:.3f} | {err_line[2]:.3f} | {1000 * t_net:.2f} ms | {sec_line:.0f} s |",
            f"| network, trained deliberately | {err_net[0]:.3f} | {err_net[1]:.3f} | {err_net[2]:.3f} | {1000 * t_net:.2f} ms | {sec:.0f} s |",
            "", "Blind distance against the furthest the car got from the centreline: "
            + ", ".join(f"{100 * d:.1f} cm -> {100 * e:.2f} cm" for d, e in sorted(loop)),
            "", "Weights in " + ", ".join(str(b) for b in bits) + " bits: mean error " + ", ".join(f"{e:.3f}" for e in err_bits),
            "", "The frames are synthetic; the car's wheelbase, steering limit and track are assumed."]
    out = ["# Ex_11.1 — Steering from a Frame", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex11.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex11.1_report.md into Ex11.1_report.pdf, with any figure
# saved as Ex11.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex11.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex11.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex11.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex11.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP11.1A · Where will the car be?

**The problem.** The policy of this set turns an image into a steering command. Where the car
then goes depends on its kinematics: the steering angle and the speed set how
the heading turns. Here the task is to predict the car's position and heading
one second ahead from its commands, with a network that knows the kinematics.

| | |
|---|---|
| track | the vendor's 3 × 2 m map, as a stadium: 1.4 m straights, turns of 0.7 m radius, a 7.20 m lap |
| car | the kinematic bicycle model; wheelbase 0.20 m and steering limit 30° assumed until measured on the car |
| driving | 1.0 m/s; a pure-pursuit controller at 30 Hz produces the commands, as a policy would |
| horizon | one second ahead, from any instant of a 15 s run |

The car has no encoder, so the ground truth is simulated; the wheelbase and the
steering limit are measured on the car before the data is built.

**What you build.** A network for $x, y$ and the heading over the next second, with the commands as
input, constrained by the bicycle model through its residual. You use it to
predict the blind distance and the margin to the track edge in each turn.

**The ground truth you get.** `MP11.1A_truth.npz`, from `tools/miniprojects/ex111_truth.py`: the bicycle
model integrated at a tolerance of 1e-10 (1e-06 and 1e-08 agree to under
0.001 mm), states and commands at 30 Hz, and the centreline.

**Worked example.** The controller holds the car within 1.69 cm of the centreline, steering
between -0.7° and 16.9°. One second ahead from $t$ = 5 s, a wheelbase just 5 %
wrong puts the prediction **2.82 cm** off - the model has to be right, not
only the network.

**Requirements.**

* Position within **2 cm** and heading within **2°**, one second ahead, from any instant.
* The prediction for one second in under **5 ms** on the car's Nano - it must run inside the 30 Hz loop.

**What you hand in.** The prediction error against the ground truth over the horizon, the predicted
turn margins, the timing on the car, and where the prediction breaks down.

### MP11.1B · How late does the car react?

**The problem.** Between the camera frame and the wheels turning there is a delay, and the
steering servo does not jump to its command but follows it with a lag. Both
set how fast the car can take a turn. From logged commands and the heading the
car actually took, find the delay and the lag.

| | |
|---|---|
| loop | 30 Hz; the command reaches the servo after a delay, the servo follows with a first-order lag - both hidden |
| records | commands and heading at 30 Hz for 15 s, heading noise 0.5° |
| laps | a smooth lap, and a lap with a 0.8 Hz weave of ±10° added to the commands |

The hidden values in the worked example are assumed - 0.08 s and 0.05 s - and
are replaced by measured ones before the data is built.

**Why a plain PINN is not enough.** A delay and a lag both make the car turn late, so a smooth lap can trade one
for the other, and a fit will still return two confident numbers.

**What you build.** An inverse model: the bicycle kinematics with a trainable delay and servo lag,
fitted to the logs. You show which lap separates the two.

**The ground truth you get.** `MP11.1B_readings.npz` to fit and `MP11.1B_truth.npz` with the hidden
values, from the same script.

**Worked example.** Adding 0.02 s to the delay and taking 0.02 s off the lag changes the heading
by at most **4.44°** on the smooth lap, but by **27.11°** on the weaving one:
the weave is what tells delay and lag apart.

**Requirements.**

* The delay and the lag each within **10 ms**, from the weaving lap.
* From the smooth lap alone, the combination that can be recovered, and the one that cannot.
* The top speed through a 0.7 m turn that your delay and lag allow.

**What you hand in.** The recovered delay and lag against the true ones, from each lap, the manoeuvre
that separates them, and the top speed they allow in the tightest turn.